# LiDAR Gap Analysis — Puerto Rico G-LiHT
**Luquillo Experimental Forest** | Epochs: March 2017 (pre-Maria) · May 2018 (post-Maria) · March 2020 (recovery)

Pipeline:
1. Load & align LAS files across epochs
2. Compute absolute C2C nearest-neighbour distances
3. Extract connected components to isolate damage clumps
4. Power-law gap size distribution analysis
5. Epoch comparison (damage vs recovery)

In [ ]:
import os
import re
import csv
import sys
import time
import gzip
import shutil
import tempfile
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

import cloudComPy as cc
cc.initCC()

# Make sure the src/ module is importable
sys.path.insert(0, str(Path('src').resolve()))
from connected_components import run_connected_component_analysis

print('[INIT] CloudComPy initialized')

## 1. Configuration
All paths and analysis parameters. Edit this cell before running.

In [ ]:
# ── Input directories ────────────────────────────────────────────────────────
DIR_2017 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_11March2017_FIA12/lidar/las')
DIR_2018 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_2May018_FIA12/lidar/las')
DIR_2020 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_15March2020_FIA12/lidar/las')

# ── Output directories ───────────────────────────────────────────────────────
OUT_DIR  = Path('/home/ogh6/Downloads/raw_c2c_output')
CSV_DIR  = OUT_DIR / 'csv'
PLOT_DIR = OUT_DIR / 'plots'
BIN_DIR  = OUT_DIR / 'bin'

for d in [OUT_DIR, CSV_DIR, PLOT_DIR, BIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Processing parameters ────────────────────────────────────────────────────
USE_SUBSAMPLE  = True
SUBSAMPLE_N    = 5_000_000
THREADS        = 12

# ── C2C analysis parameters ──────────────────────────────────────────────────
NOISE_CEIL_M   = 20.0
C2C_THRESHOLDS = [2, 5, 10, 15]
EXTREME_PCTILE = 95
HIST_MAX_M     = 20.0
HIST_BINS      = 200

# ── Connected component parameters ───────────────────────────────────────────
CC_CONFIG = {
    'damage_threshold':    2.0,    # metres — Leitold et al.
    'recovery_threshold':  2.0,
    'min_component_pts':   100,
    'min_gap_area_m2':     10.0,   # Gorgens et al.
    'overlap_min_fraction': 0.80,
    'overlap_buffer_m':    50.0,
    'max_components':      100_000,
}

GRID_RE  = re.compile(r'(l\d+s\d+)', re.IGNORECASE)
_TMP_DIR = Path(tempfile.gettempdir()) / f'cc_c2c_{int(time.time())}'
_TMP_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  2017 dir : {DIR_2017}')
print(f'  2018 dir : {DIR_2018}')
print(f'  2020 dir : {DIR_2020}')
print(f'  Output   : {OUT_DIR}')

## 2. Utility Functions

In [ ]:
def tnow(): return time.time()

def fmt_s(dt):
    if dt < 60:   return f'{dt:.1f}s'
    if dt < 3600: return f'{dt/60:.1f}m'
    return f'{dt/3600:.2f}h'

def mem_gb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    except ImportError:
        return -1.0

def gunzip_if_needed(p: Path) -> Path:
    if p.suffix != '.gz':
        return p
    out = _TMP_DIR / p.name[:-3]
    if out.exists() and out.stat().st_size > 0:
        return out
    t0 = tnow()
    print(f'  [UNZIP] {p.name} ...', flush=True)
    with gzip.open(p, 'rb') as fin, open(out, 'wb') as fout:
        shutil.copyfileobj(fin, fout)
    print(f'  [UNZIP] done {fmt_s(tnow()-t0)}', flush=True)
    return out

def grid_id_from_name(p: Path) -> str:
    m = GRID_RE.search(p.name)
    if not m:
        raise ValueError(f'No grid ID in {p.name}')
    return m.group(1).lower()

def find_match(folder: Path, grid: str):
    for f in folder.glob('*.las*'):
        m = GRID_RE.search(f.name)
        if m and m.group(1).lower() == grid:
            return f
    return None

def load_cloud(p: Path, shift=None):
    t0 = tnow()
    print(f'  [LOAD] {p.name} ...', flush=True)
    if shift is not None:
        cloud = cc.loadPointCloud(
            filename=str(p), mode=cc.CC_SHIFT_MODE.XYZ,
            x=shift[0], y=shift[1], z=shift[2]
        )
    else:
        cloud = cc.loadPointCloud(str(p))
    if cloud is None:
        raise RuntimeError(f'Failed to load {p}')
    applied_shift = (0.0, 0.0, 0.0)
    if cloud.isShifted():
        gs = cloud.getGlobalShift()
        applied_shift = (float(gs[0]), float(gs[1]), float(gs[2]))
    print(f'  [LOAD] done {fmt_s(tnow()-t0)} | pts={cloud.size():,} '
          f'| shift=({applied_shift[0]:.1f},{applied_shift[1]:.1f},{applied_shift[2]:.1f}) '
          f'| mem={mem_gb():.2f} GB', flush=True)
    return cloud, applied_shift

def safe_delete(entity):
    try:
        if entity is not None:
            cc.deleteEntity(entity)
    except Exception:
        pass

print('Utility functions ready.')

## 3. Normalization Check

In [ ]:
def check_normalization(cloud, label=''):
    bb    = cloud.getOwnBB()
    z_min = float(bb.minCorner()[2])
    z_max = float(bb.maxCorner()[2])
    sf_dic    = cloud.getScalarFieldDic()
    sf_names  = list(sf_dic.keys())
    has_class = any('classif' in n.lower() or 'class' in n.lower() for n in sf_names)

    if z_min > 50:
        normalised = False
        note = f'Z min={z_min:.1f}m — raw GPS elevation. C2C distances still valid.'
    elif -2 <= z_min < 5:
        normalised = True
        note = 'Appears height-normalised.'
    else:
        normalised = None
        note = f'Ambiguous (z_min={z_min:.1f}). Check manually.'

    return {'label': label, 'z_min': z_min, 'z_max': z_max,
            'has_classification': has_class, 'likely_normalised': normalised,
            'note': note, 'scalar_fields': ', '.join(sf_names)}

def print_norm_report(info):
    status = {True: 'YES', False: 'NO', None: 'UNCLEAR'}[info['likely_normalised']]
    print(f"  [NORM] {info['label']}: z_min={info['z_min']:.1f} "
          f"z_max={info['z_max']:.1f} | normalised={status}")
    if info['likely_normalised'] is False:
        print(f"         NOTE: {info['note']}")

print('Normalization functions ready.')

## 4. Subsampling

In [ ]:
def subsample_cloud(cloud, target_n: int):
    if cloud.size() <= target_n:
        return cloud
    t0 = tnow()
    print(f'  [SUB] {cloud.size():,} -> ~{target_n:,} ...', flush=True)
    ref = cc.CloudSamplingTools.subsampleCloudWithOctree(
        cloud, int(target_n),
        cc.SUBSAMPLING_CELL_METHOD.NEAREST_POINT_TO_CELL_CENTER,
    )
    sub, res = cloud.partialClone(ref)
    if res != 0:
        raise RuntimeError(f'partialClone failed (code {res})')
    print(f'  [SUB] done {fmt_s(tnow()-t0)} | out={sub.size():,} | mem={mem_gb():.2f} GB', flush=True)
    return sub

print('Subsampling functions ready.')

## 5. C2C Distance Computation
Two-pass approach (approx → best octree level → exact multi-threaded). `maxSearchDist` is intentionally left at default (-1) to preserve multi-threading.

In [ ]:
def compute_c2c(src, tgt, sf_name, threads=12):
    t0 = tnow()
    print(f'  [C2C] {sf_name}: computing approx distances ...', flush=True)

    approx_stats = cc.DistanceComputationTools.computeApproxCloud2CloudDistance(src, tgt)
    if approx_stats:
        print(f'  [C2C] approx stats: min={approx_stats[0]:.2f} max={approx_stats[1]:.2f} mean={approx_stats[2]:.2f}', flush=True)

    best_level = cc.DistanceComputationTools.determineBestOctreeLevel(src, None, tgt)
    print(f'  [C2C] octree level = {best_level}', flush=True)

    params = cc.Cloud2CloudDistancesComputationParams()
    params.octreeLevel    = best_level
    params.multiThread    = True
    params.maxThreadCount = int(threads)

    print(f'  [C2C] {sf_name}: computing exact distances (threads={threads}) ...', flush=True)
    ret = cc.DistanceComputationTools.computeCloud2CloudDistances(src, tgt, params)
    if ret < 0:
        print(f'  [C2C] WARNING: computeCloud2CloudDistances returned {ret}', flush=True)

    n_sf = src.getNumberOfScalarFields()
    sf   = src.getScalarField(n_sf - 1)
    sf.setName(sf_name)
    sf.computeMinAndMax()

    # Remove the approx SF
    sf_dic = src.getScalarFieldDic()
    for name in sf_dic:
        if 'approx' in name.lower():
            src.deleteScalarField(sf_dic[name])
            print(f"  [C2C] removed '{name}' SF", flush=True)
            break

    print(f'  [C2C] {sf_name} done in {fmt_s(tnow()-t0)}', flush=True)
    return sf_name


def get_sf_as_array(cloud, sf_name):
    sf = cloud.getScalarField(sf_name)
    if sf is None:
        sf_dic = cloud.getScalarFieldDic()
        if sf_name in sf_dic:
            sf = cloud.getScalarField(sf_dic[sf_name])
        if sf is None:
            raise KeyError(f"SF '{sf_name}' not found. Available: {list(cloud.getScalarFieldDic().keys())}")
    return np.asarray(sf.toNpArrayCopy(), dtype=np.float64)

print('C2C functions ready.')

## 6. Statistical Analysis Functions

In [ ]:
def c2c_summary_stats(c2c, label=''):
    clean   = c2c[c2c <= NOISE_CEIL_M]
    n_total = len(c2c)
    n_clean = len(clean)
    n_noise = n_total - n_clean
    out = {'label': label, 'n_total': n_total, 'n_clean': n_clean,
           'n_noise': n_noise, 'pct_noise': n_noise / max(n_total, 1) * 100}
    if n_clean > 0:
        out.update({
            'mean':   float(np.mean(clean)),   'median': float(np.median(clean)),
            'std':    float(np.std(clean)),     'min':    float(np.min(clean)),
            'max':    float(np.max(clean)),     'p05':    float(np.percentile(clean, 5)),
            'p25':    float(np.percentile(clean, 25)), 'p75': float(np.percentile(clean, 75)),
            'p95':    float(np.percentile(clean, 95)), 'p99': float(np.percentile(clean, 99)),
        })
    return out

def c2c_threshold_fractions(c2c, thresholds=C2C_THRESHOLDS):
    clean = c2c[c2c <= NOISE_CEIL_M]
    n = max(len(clean), 1)
    return {t: float(np.sum(clean >= t) / n) for t in thresholds}

def extreme_value_report(c2c, label='', percentile=EXTREME_PCTILE):
    clean = c2c[c2c <= NOISE_CEIL_M]
    if len(clean) == 0:
        return {'label': label, 'error': 'no clean points'}
    thresh   = float(np.percentile(clean, percentile))
    extreme  = clean[clean > thresh]
    return {
        'label': label, 'percentile': percentile, 'threshold_m': thresh,
        'n_extreme': len(extreme), 'n_clean': len(clean),
        'extreme_mean':   float(np.mean(extreme)),
        'extreme_median': float(np.median(extreme)),
        'extreme_std':    float(np.std(extreme)),
        'extreme_max':    float(np.max(extreme)),
        'pct_2_5m':  float(np.sum((extreme >= 2)  & (extreme < 5))  / max(len(extreme), 1) * 100),
        'pct_5_10m': float(np.sum((extreme >= 5)  & (extreme < 10)) / max(len(extreme), 1) * 100),
        'pct_10_15m':float(np.sum((extreme >= 10) & (extreme < 15)) / max(len(extreme), 1) * 100),
        'pct_15_20m':float(np.sum((extreme >= 15) & (extreme < 20)) / max(len(extreme), 1) * 100),
    }

def statistical_comparison(c2c_a, c2c_b, label_a='A', label_b='B'):
    a = c2c_a[c2c_a <= NOISE_CEIL_M]
    b = c2c_b[c2c_b <= NOISE_CEIL_M]
    ks_stat, ks_p = sp_stats.ks_2samp(a, b)
    max_n = 500_000
    a_t = np.random.choice(a, min(len(a), max_n), replace=False) if len(a) > max_n else a
    b_t = np.random.choice(b, min(len(b), max_n), replace=False) if len(b) > max_n else b
    mw_stat, mw_p = sp_stats.mannwhitneyu(a_t, b_t, alternative='two-sided')
    result = {
        'comparison': f'{label_a} vs {label_b}',
        f'{label_a}_n': len(a), f'{label_b}_n': len(b),
        'ks_statistic': float(ks_stat), 'ks_pvalue': float(ks_p),
        'mannwhitney_U': float(mw_stat), 'mannwhitney_pvalue': float(mw_p),
    }
    for tag, arr in [(label_a, a), (label_b, b)]:
        result[f'{tag}_mean']   = float(np.mean(arr))
        result[f'{tag}_median'] = float(np.median(arr))
        result[f'{tag}_std']    = float(np.std(arr))
    return result

print('Statistical analysis functions ready.')

## 7. C2C Plotting

In [ ]:
def plot_grid(grid_id, c2c_1718, c2c_1820, out_dir=PLOT_DIR):
    clean_1718 = c2c_1718[c2c_1718 <= NOISE_CEIL_M]
    clean_1820 = c2c_1820[c2c_1820 <= NOISE_CEIL_M]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Grid {grid_id.upper()} — Raw C2C Displacement Analysis', fontsize=14, fontweight='bold')

    edges = np.linspace(0, HIST_MAX_M, HIST_BINS + 1)
    ax = axes[0, 0]
    ax.hist(clean_1718, bins=edges, density=True, alpha=0.5, color='red',  label='2017→2018 (Maria)')
    ax.hist(clean_1820, bins=edges, density=True, alpha=0.5, color='blue', label='2018→2020 (Recovery)')
    ax.set_xlabel('C2C distance (m)'); ax.set_ylabel('Density')
    ax.set_title('(a) C2C Distribution'); ax.legend(fontsize=8); ax.set_xlim(0, HIST_MAX_M)

    ax = axes[0, 1]
    for arr, lbl, clr in [(clean_1718, '2017→2018', 'red'), (clean_1820, '2018→2020', 'blue')]:
        s = np.sort(arr); cdf = np.arange(1, len(s)+1)/len(s)
        step = max(len(s)//10000, 1)
        ax.plot(s[::step], cdf[::step], color=clr, linewidth=1.5, label=lbl)
    ax.set_xlabel('C2C distance (m)'); ax.set_ylabel('Cumulative proportion')
    ax.set_title('(b) CDF'); ax.legend(fontsize=8); ax.set_xlim(0, HIST_MAX_M); ax.grid(True, alpha=0.3)

    ax = axes[0, 2]
    x_pos = np.arange(len(C2C_THRESHOLDS)); width = 0.35
    frac_1718 = c2c_threshold_fractions(c2c_1718)
    frac_1820 = c2c_threshold_fractions(c2c_1820)
    ax.bar(x_pos-width/2, [frac_1718[t]*100 for t in C2C_THRESHOLDS], width, color='red',  alpha=0.7, label='2017→2018')
    ax.bar(x_pos+width/2, [frac_1820[t]*100 for t in C2C_THRESHOLDS], width, color='blue', alpha=0.7, label='2018→2020')
    ax.set_xticks(x_pos); ax.set_xticklabels([f'>={t}m' for t in C2C_THRESHOLDS])
    ax.set_ylabel('% of clean points'); ax.set_title('(c) Threshold Exceedance'); ax.legend(fontsize=8)

    ax = axes[1, 0]
    bands = ['2-5m', '5-10m', '10-15m', '15-20m']
    ev_1718 = extreme_value_report(c2c_1718); ev_1820 = extreme_value_report(c2c_1820)
    keys = ['pct_2_5m', 'pct_5_10m', 'pct_10_15m', 'pct_15_20m']
    ax.bar(np.arange(4)-width/2, [ev_1718.get(k,0) for k in keys], width, color='red',  alpha=0.7, label='2017→2018')
    ax.bar(np.arange(4)+width/2, [ev_1820.get(k,0) for k in keys], width, color='blue', alpha=0.7, label='2018→2020')
    ax.set_xticks(np.arange(4)); ax.set_xticklabels(bands)
    ax.set_ylabel(f'% of extreme pts (>{EXTREME_PCTILE}th pctile)'); ax.set_title('(d) Extreme Value Bands'); ax.legend(fontsize=8)

    ax = axes[1, 1]
    fine = np.linspace(0, 5, 100)
    ax.hist(clean_1718, bins=fine, density=True, alpha=0.5, color='red',  label='2017→2018')
    ax.hist(clean_1820, bins=fine, density=True, alpha=0.5, color='blue', label='2018→2020')
    ax.set_xlabel('C2C distance (m)'); ax.set_ylabel('Density')
    ax.set_title('(e) Detail: 0–5 m'); ax.legend(fontsize=8); ax.axvline(2.0, color='gray', linestyle='--', alpha=0.5)

    ax = axes[1, 2]; ax.axis('off')
    s1 = c2c_summary_stats(c2c_1718); s2 = c2c_summary_stats(c2c_1820)
    txt = (f"2017→2018 (Hurricane Maria)\n"
           f"  n={s1['n_clean']:,}  noise={s1['n_noise']:,} ({s1['pct_noise']:.1f}%)\n"
           f"  mean={s1.get('mean',0):.2f}  med={s1.get('median',0):.2f}  std={s1.get('std',0):.2f}\n"
           f"  p95={s1.get('p95',0):.2f}  p99={s1.get('p99',0):.2f}\n\n"
           f"2018→2020 (Recovery)\n"
           f"  n={s2['n_clean']:,}  noise={s2['n_noise']:,} ({s2['pct_noise']:.1f}%)\n"
           f"  mean={s2.get('mean',0):.2f}  med={s2.get('median',0):.2f}  std={s2.get('std',0):.2f}\n"
           f"  p95={s2.get('p95',0):.2f}  p99={s2.get('p99',0):.2f}\n\n"
           f"NOISE CEILING: {NOISE_CEIL_M}m\nERROR FLOOR: 2m")
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title('(f) Summary Statistics')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    outpath = out_dir / f'{grid_id}_c2c_analysis.png'
    fig.savefig(str(outpath), dpi=150); plt.close(fig)
    print(f'  [PLOT] saved {outpath.name}', flush=True)
    return outpath

def write_csv(filepath, rows, fieldnames=None):
    if not rows: return
    if fieldnames is None:
        seen = {}
        for r in rows:
            for k in r.keys():
                if k not in seen: seen[k] = True
        fieldnames = list(seen.keys())
    with open(filepath, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
    print(f'  [CSV] {filepath}', flush=True)

print('Plotting and CSV functions ready.')

## 8. Determine Global Coordinate Shift
Load one file to read the auto-shift, then force that shift on every subsequent file so all epochs share the same coordinate frame.

In [ ]:
files_2017 = sorted(DIR_2017.glob('*.las*'))
print(f'Found {len(files_2017)} files in 2017 directory')

global_shift = None
for probe_file in files_2017:
    try:
        grid_id_from_name(probe_file)
    except ValueError:
        continue
    p_probe = gunzip_if_needed(probe_file)
    print(f'[SHIFT] Probing {p_probe.name} ...')
    probe_cloud = cc.loadPointCloud(str(p_probe))
    if probe_cloud is None:
        continue
    global_shift = tuple(float(x) for x in probe_cloud.getGlobalShift()) if probe_cloud.isShifted() else (0.0, 0.0, 0.0)
    bb = probe_cloud.getOwnBB()
    print(f'[SHIFT] Global shift = {global_shift}')
    print(f'[SHIFT] Shifted BB: X=[{float(bb.minCorner()[0]):.1f}, {float(bb.maxCorner()[0]):.1f}] '
          f'Y=[{float(bb.minCorner()[1]):.1f}, {float(bb.maxCorner()[1]):.1f}] '
          f'Z=[{float(bb.minCorner()[2]):.1f}, {float(bb.maxCorner()[2]):.1f}]')
    safe_delete(probe_cloud)
    break

if global_shift is None:
    raise RuntimeError('Could not determine coordinate shift from any 2017 file')
print(f'\nUsing global shift {global_shift} for ALL files.')

## 9. Main Processing Loop
For each matched grid tile across all three epochs:
- Load & subsample
- Check overlap & clip if needed
- Compute C2C (2017→2018 and 2018→2020)
- Run connected component gap analysis
- Save statistics and plots

In [ ]:
all_summaries  = []
all_thresholds = []
all_tests      = []
all_extreme    = []
all_norm       = []
all_cc_results = []

n_processed = n_skip_2018 = n_skip_2020 = 0
norm_done   = False

for idx, f2017 in enumerate(files_2017, start=1):
    loop_t0 = tnow()
    try:
        grid = grid_id_from_name(f2017)
    except ValueError:
        continue

    f2018 = find_match(DIR_2018, grid)
    if f2018 is None: n_skip_2018 += 1; continue
    f2020 = find_match(DIR_2020, grid)
    if f2020 is None: n_skip_2020 += 1; continue

    print(f'\n{"="*70}')
    print(f'  [{idx}/{len(files_2017)}]  GRID {grid.upper()}')
    print(f'{"="*70}')

    p2017 = gunzip_if_needed(f2017)
    p2018 = gunzip_if_needed(f2018)
    p2020 = gunzip_if_needed(f2020)

    c2017, _ = load_cloud(p2017, shift=global_shift)
    c2018, _ = load_cloud(p2018, shift=global_shift)
    c2020, _ = load_cloud(p2020, shift=global_shift)

    # Bounding box sanity check
    for lbl, c in [('2018', c2018), ('2020', c2020)]:
        bb_ref = c2017.getOwnBB(); bb_oth = c.getOwnBB()
        dist = (abs(float(bb_ref.minCorner()[0]) - float(bb_oth.minCorner()[0])) +
                abs(float(bb_ref.minCorner()[1]) - float(bb_oth.minCorner()[1])))
        tag = 'OK' if dist <= 100 else 'WARN — possible shift mismatch'
        print(f'  [CHECK] 2017 vs {lbl} BB offset: {dist:.1f}m ({tag})')

    # Normalization check (first grid only for all epochs, then 2017 only)
    if not norm_done:
        for c, lbl in [(c2017, f'{grid}_2017'), (c2018, f'{grid}_2018'), (c2020, f'{grid}_2020')]:
            info = check_normalization(c, lbl)
            print_norm_report(info); all_norm.append(info)
        norm_done = True
    else:
        info = check_normalization(c2017, f'{grid}_2017')
        all_norm.append(info)
        if info['likely_normalised'] is False:
            print(f"  [NORM] {grid}: raw GPS (z_min={info['z_min']:.0f}m)")

    # Subsample
    did_sub = False
    if USE_SUBSAMPLE:
        c2017s = subsample_cloud(c2017, SUBSAMPLE_N)
        c2018s = subsample_cloud(c2018, SUBSAMPLE_N)
        c2020s = subsample_cloud(c2020, SUBSAMPLE_N)
        did_sub = (c2017s is not c2017)
    else:
        c2017s = c2017; c2018s = c2018; c2020s = c2020

    # C2C: 2017 → 2018
    sf_1718  = compute_c2c(c2017s, c2018s, 'C2C_2017_2018', threads=THREADS)
    c2c_1718 = get_sf_as_array(c2017s, sf_1718)
    cc.SavePointCloud(c2017s, str(BIN_DIR / f'{grid}_2017_c2c.bin'))
    print(f'  [DATA] 1718: {len(c2c_1718):,} pts extracted')

    if did_sub and c2017s is not c2017: safe_delete(c2017s); c2017s = None
    safe_delete(c2017); c2017 = None
    print(f'  [MEM] freed 2017 clouds | mem={mem_gb():.2f} GB')

    # C2C: 2018 → 2020
    sf_1820  = compute_c2c(c2018s, c2020s, 'C2C_2018_2020', threads=THREADS)
    c2c_1820 = get_sf_as_array(c2018s, sf_1820)
    cc.SavePointCloud(c2018s, str(BIN_DIR / f'{grid}_2018_c2c.bin'))
    print(f'  [DATA] 1820: {len(c2c_1820):,} pts extracted')

    # Summary stats
    s1 = c2c_summary_stats(c2c_1718, f'{grid}_2017to2018'); s1['grid'] = grid; s1['interval'] = '2017-2018'
    s2 = c2c_summary_stats(c2c_1820, f'{grid}_2018to2020'); s2['grid'] = grid; s2['interval'] = '2018-2020'
    all_summaries.extend([s1, s2])

    # Threshold fractions
    tf1 = c2c_threshold_fractions(c2c_1718); tf2 = c2c_threshold_fractions(c2c_1820)
    row = {'grid': grid}
    for t in C2C_THRESHOLDS:
        row[f'1718_gte_{t}m'] = tf1[t]; row[f'1820_gte_{t}m'] = tf2[t]
    all_thresholds.append(row)

    # Statistical tests
    test = statistical_comparison(c2c_1718, c2c_1820, '2017_2018', '2018_2020')
    test['grid'] = grid; all_tests.append(test)

    # Extreme values
    ev1 = extreme_value_report(c2c_1718, f'{grid}_2017to2018'); ev1['grid'] = grid; ev1['interval'] = '2017-2018'
    ev2 = extreme_value_report(c2c_1820, f'{grid}_2018to2020'); ev2['grid'] = grid; ev2['interval'] = '2018-2020'
    all_extreme.extend([ev1, ev2])

    # C2C diagnostic plot
    plot_grid(grid, c2c_1718, c2c_1820)

    # ── Connected component gap analysis ─────────────────────────────────────
    # Re-load c2017s for connected components (was freed above)
    # We use c2018s (still in memory) which has the C2C SF attached.
    # For damage epoch we need c2017s back with its SF.
    # Reload from the saved .bin file.
    c2017s_bin = cc.loadPointCloud(str(BIN_DIR / f'{grid}_2017_c2c.bin'))
    cc_results = run_connected_component_analysis(
        c2017s_bin, c2018s, c2020s,
        c2c_1718, c2c_1820,
        grid_id=grid,
        config=CC_CONFIG,
        out_dirs={'plots': PLOT_DIR},
    )
    all_cc_results.append(cc_results)
    safe_delete(c2017s_bin)

    # Cleanup
    if did_sub: safe_delete(c2018s); safe_delete(c2020s)
    safe_delete(c2018); safe_delete(c2020)

    n_processed += 1
    print(f'\n  [DONE] {grid.upper()} in {fmt_s(tnow()-loop_t0)} | mem={mem_gb():.2f} GB')

print(f'\nLoop complete. Processed: {n_processed} | Skipped 2018: {n_skip_2018} | Skipped 2020: {n_skip_2020}')

## 10. Write CSV Outputs

In [ ]:
write_csv(CSV_DIR / 'c2c_summary_stats.csv',      all_summaries)
write_csv(CSV_DIR / 'c2c_threshold_fractions.csv', all_thresholds)
write_csv(CSV_DIR / 'c2c_statistical_tests.csv',   all_tests)
write_csv(CSV_DIR / 'c2c_extreme_values.csv',      all_extreme)
write_csv(CSV_DIR / 'normalisation_checks.csv',    all_norm)

# Flatten connected component epoch comparison results
cc_epoch_rows = []
for r in all_cc_results:
    if r.get('status') == 'ok' and 'epoch_comparison' in r:
        row = {'grid_id': r['grid_id']}
        ec = r['epoch_comparison']
        row.update({
            'n_gaps_damage':     ec.get('n_gaps_damage'),
            'n_gaps_recovery':   ec.get('n_gaps_recovery'),
            'mean_area_damage':  ec.get('mean_gap_area_damage_m2'),
            'mean_area_recovery':ec.get('mean_gap_area_recovery_m2'),
            'total_area_damage': ec.get('total_damaged_area_damage_m2'),
            'total_area_recovery':ec.get('total_damaged_area_recovery_m2'),
            'alpha_damage':      ec.get('power_law_damage', {}).get('alpha_mle'),
            'alpha_recovery':    ec.get('power_law_recovery', {}).get('alpha_mle'),
            'delta_alpha':       ec.get('power_law_comparison', {}).get('delta_alpha'),
            'ks_pvalue':         ec.get('ks_2samp', {}).get('pvalue'),
            'mw_pvalue':         ec.get('mann_whitney_u', {}).get('pvalue'),
            'pl_interpretation': ec.get('power_law_comparison', {}).get('interpretation'),
        })
        cc_epoch_rows.append(row)

write_csv(CSV_DIR / 'connected_component_epoch_comparison.csv', cc_epoch_rows)
print('\nAll CSVs written.')
print(f'Output directory: {OUT_DIR}')

## 11. Quick Results Review
Run this cell after the pipeline to see a summary of findings across all grids.

In [ ]:
import pandas as pd

summary_df = pd.read_csv(CSV_DIR / 'c2c_summary_stats.csv')
cc_df      = pd.read_csv(CSV_DIR / 'connected_component_epoch_comparison.csv')

print('=== C2C Summary (damage epoch 2017→2018) ===')
damage = summary_df[summary_df['interval'] == '2017-2018']
print(damage[['grid', 'n_clean', 'mean', 'median', 'p95']].to_string(index=False))

print('\n=== Power Law Exponents ===')
print(cc_df[['grid_id', 'alpha_damage', 'alpha_recovery', 'delta_alpha', 'pl_interpretation']].to_string(index=False))

print('\n=== Gap Counts ===')
print(cc_df[['grid_id', 'n_gaps_damage', 'n_gaps_recovery', 'mean_area_damage', 'mean_area_recovery']].to_string(index=False))